In [ ]:
#| label: fig2cell
from pathlib import Path
import numpy as np, pandas as pd
from plotly.subplots import make_subplots
from scipy.stats import linregress

PXIN, PT = 96, 96 / 72
px = lambda i: round(i * PXIN); pt = lambda p: p * PT
DATA = next(p for p in [Path('dashboard_standard_reg'), Path('../dashboard_standard_reg')] if p.exists())

T1_SCALE = 1000.0
merged = pd.read_csv(DATA / 'data_merged.csv')
t1_cols = ['T1_mean_05', 'T1_mean_510', 'T1_mean_1015', 'T1_mean_015']
merged[t1_cols] *= T1_SCALE
T1_BANDS = [('T1_mean_015', '0–15 mm', '#2B2D42'), ('T1_mean_05', '0–5 mm', '#3B82F6'),
            ('T1_mean_510', '5–10 mm', '#228B5E'), ('T1_mean_1015', '10–15 mm', '#8B5CF6')]
PANELS = [('Macula', 'All_1_3_gcc', 'GCC thickness (µm)', 'GCC All (1–3 mm)'),
          ('Optic Disc', 'All_um_', 'RNFL thickness (µm)', 'RNFL Average')]
AX = dict(color='black', linecolor='black', showline=True, mirror=False, showgrid=False,
          zeroline=False, ticks='outside', tickcolor='black', fixedrange=True)


def bh_fdr(p):
    p = np.asarray(p, float); out = np.full(p.shape, np.nan)
    ok = np.where(~np.isnan(p))[0]; m = len(ok); order = ok[np.argsort(p[ok])]; prev = 1.0
    for k in range(m - 1, -1, -1):
        prev = min(prev, p[order[k]] * m / (k + 1)); out[order[k]] = prev
    return out


fmt = lambda v: f'{v:.2f}'.replace('0.', '.')

HL = dict(bgcolor='#222', font=dict(color='#fff'))   # dark hover popup, like the dashboard

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=[f'{t} – {o}' for t, _, _, o in PANELS])
for col, (title, sector, xlab, octName) in enumerate(PANELS, start=1):
    leg = 'legend' if col == 1 else 'legend2'
    dfs = [(merged[[sector, band, 'Eye', 'MRI_ID']].dropna(), band) for band, _, _ in T1_BANDS]
    fits = [(d, linregress(d[sector], d[band]) if len(d) >= 5 else None) for d, band in dfs]
    pfdr = bh_fdr([f.pvalue if f else np.nan for _, f in fits])
    for (band, lbl, c), (d, f), q in zip(T1_BANDS, fits, pfdr):
        if f is None:
            continue
        vis = True if band == 'T1_mean_015' else 'legendonly'
        grp = f'{col}-{lbl}'
        for eye in ('OD', 'OS'):
            g = d[d.Eye == eye]
            marker = (dict(color='white', size=10, line=dict(color=c, width=1.5)) if eye == 'OD'
                      else dict(color=c, size=10, line=dict(width=0)))
            hov = (f'<b>%{{customdata}}</b> · {eye} · {lbl} <br>'
                   f'{octName} = %{{x:.2f}}<br>T₁ = %{{y:.0f}} ms<extra></extra>')
            fig.add_scatter(x=g[sector], y=g[band], customdata=g['MRI_ID'], mode='markers',
                legend=leg, legendgroup=grp, showlegend=False, visible=vis, row=1, col=col,
                marker=marker, hovertemplate=hov, hoverlabel=HL)
        xs = np.array([d[sector].min(), d[sector].max()])
        star = ' *' if (not np.isnan(q) and q < 0.05) else ''
        fig.add_scatter(x=xs, y=f.intercept + f.slope * xs, mode='lines', legend=leg, legendgroup=grp,
            visible=vis, row=1, col=col, line=dict(color=c, width=2), hoverinfo='skip',
            name=f'{lbl}  (R²={fmt(f.rvalue ** 2)}){star}')
    fig.update_xaxes(title=dict(text=xlab, standoff=5), row=1, col=col, **AX)
    fig.update_yaxes(title=dict(text='ON T₁ (ms)'), row=1, col=col, **AX)

LEG = dict(bgcolor='rgba(255,255,255,0.7)', bordercolor='#ccc', borderwidth=1,
           font=dict(size=pt(9)), xanchor='right', yanchor='top', y=0.99,
           groupclick='togglegroup', tracegroupgap=1)
fig.update_layout(autosize=True, paper_bgcolor='white', plot_bgcolor='white', dragmode=False,
    font=dict(color='black', family='DejaVu Sans, Arial, sans-serif', size=pt(12)),
    margin=dict(l=px(0.9), r=px(0.1), t=px(0.5), b=px(0.6)),
    legend=dict(**LEG, x=0.43), legend2=dict(**LEG, x=0.99))
for a in fig.layout.annotations:
    a.font.size = pt(14)
fig.show(config={"responsive": True, "edits": {"legendPosition": True}})